# M3.S1 — Performance Models, Scalability & Parallel Patterns
## Interactive HPC notebook

This notebook accompanies **M3.S1 — Performance Models, Scalability & Parallel Patterns**.

The goal is to connect the slides to small experiments you can **predict, run, observe and explain**.

### What you will do

1. calculate **speedup** and **efficiency** from measured runtimes;
2. see why an apparently small **serial fraction** limits scalability;
3. compare **Amdahl** and **Gustafson** as answers to different questions;
4. experience the difference between **strong scaling** and **weak scaling**;
5. act as a **scaling detective** and diagnose different curve shapes;
6. match common **parallel patterns** to real HPC problems;
7. correct a misleading performance claim using evidence.

### Classroom method

> **PREDICT → RUN → OBSERVE → EXPLAIN**

### Important

Some experiments use short `sleep()` calls to make scaling behaviour visible and reproducible in a teaching notebook. They are **conceptual timing experiments, not hardware benchmarks**.

For real performance claims, use controlled measurements on the HPC compute nodes.

> **Notebook build: M3S1-2026-09-26-v1**
>
> Answers are included in **Reveal answer** sections. Try each activity before opening them.

# 0 — Before you start

For every exercise:

1. **Predict** what you expect.
2. **Run** the code.
3. **Observe** the result.
4. **Explain** it in one sentence.

Do not rush to the answer. The prediction is part of the exercise.

In [ ]:
import math
import statistics
import time
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt

print("Notebook ready.")

# 1 — Speedup and efficiency from real timings
### How much useful performance did the extra cores buy?

A program was measured with the following runtimes:

| Cores | Runtime |
|---:|---:|
| 1 | 80 s |
| 2 | 42 s |
| 4 | 23 s |
| 8 | 14 s |
| 16 | 11 s |

### Predict

Before running the next cell:

1. Is the 8-core speedup closer to **4×**, **6×**, or **8×**?
2. Will efficiency stay near 100% as we add cores?
3. At what point do you expect diminishing returns to become obvious?

Recall:

\[
S(P)=\frac{T_1}{T_P}
\]

\[
E(P)=\frac{S(P)}{P}
\]

In [ ]:
cores = [1, 2, 4, 8, 16]
times = [80.0, 42.0, 23.0, 14.0, 11.0]

baseline = times[0]
speedups = [baseline / t for t in times]
efficiencies = [s / p for s, p in zip(speedups, cores)]

print(f"{'Cores':>5} {'Time(s)':>8} {'Speedup':>9} {'Efficiency':>12}")
for p, t, s, e in zip(cores, times, speedups, efficiencies):
    print(f"{p:5d} {t:8.1f} {s:9.2f} {100*e:11.1f}%")

plt.figure(figsize=(7, 4))
plt.plot(cores, speedups, marker='o', label='Observed speedup')
plt.plot(cores, cores, linestyle='--', label='Ideal speedup')
plt.xlabel('Cores')
plt.ylabel('Speedup')
plt.title('Observed vs ideal scaling')
plt.xticks(cores)
plt.grid(alpha=0.25)
plt.legend()
plt.show()

### Try one change

Change the 8-core runtime from `14.0` to `18.0` in the previous cell and run it again.

- What happens to 8-core speedup?
- What happens to efficiency?
- Does adding cores still guarantee proportional performance?

<details>
<summary><strong>Reveal answer</strong></summary>

With the original data, 8 cores give a speedup of about **5.7×**, not 8×, and efficiency is about **71%**.

The useful question is not only *“Is it faster?”* but also *“How effectively are the extra resources being converted into performance?”*

If the 8-core runtime becomes 18 s, both speedup and efficiency fall. More resources were used, but they produced less useful acceleration.
</details>

# 2 — Amdahl's Law: the serial fraction wins

Suppose **95% of a program can run in parallel** and **5% remains serial**.

### Predict

1. With 16 cores, will the program be close to 16× faster?
2. If we had infinitely many cores, could speedup become infinite?
3. Which matters more at large core counts: the 95% parallel part or the 5% serial part?

Amdahl's model:

\[
S(P)=\frac{1}{f_s + \frac{1-f_s}{P}}
\]

where `f_s` is the serial fraction.

In [ ]:
serial_fraction = 0.05
core_counts = [1, 2, 4, 8, 16, 32, 64, 128]

def amdahl_speedup(p, serial):
    return 1.0 / (serial + (1.0 - serial) / p)

amdahl_values = [amdahl_speedup(p, serial_fraction) for p in core_counts]

print(f"Serial fraction: {serial_fraction:.1%}")
print(f"Theoretical maximum speedup as cores -> infinity: {1/serial_fraction:.1f}x")
print(f"Speedup on 16 cores: {amdahl_speedup(16, serial_fraction):.2f}x")
print(f"Speedup on 64 cores: {amdahl_speedup(64, serial_fraction):.2f}x")

plt.figure(figsize=(7, 4))
plt.plot(core_counts, amdahl_values, marker='o', label='Amdahl speedup')
plt.plot(core_counts, core_counts, linestyle='--', label='Ideal')
plt.xlabel('Cores')
plt.ylabel('Speedup')
plt.title('A small serial fraction bends the scaling curve')
plt.xscale('log', base=2)
plt.xticks(core_counts, core_counts)
plt.grid(alpha=0.25)
plt.legend()
plt.show()

### Change the model

Edit only this line:

```python
serial_fraction = 0.05
```

Try:

- `0.01` → 1% serial
- `0.10` → 10% serial
- `0.20` → 20% serial

### Explain

- Which change has the biggest effect at high core counts?
- Why can optimizing a small serial section matter so much?

<details>
<summary><strong>Reveal answer</strong></summary>

With a fixed problem size, the parallel part becomes faster as more cores are added, so the **serial fraction becomes increasingly dominant**.

The asymptotic maximum speedup is approximately `1 / serial_fraction`:

- 1% serial → at most ~100×
- 5% serial → at most ~20×
- 10% serial → at most ~10×
- 20% serial → at most ~5×

Amdahl is therefore a useful warning against assuming that more cores automatically produce proportional speedup.
</details>

# 3 — Amdahl vs Gustafson: two different questions

Amdahl asks:

> **If the problem stays fixed, how much faster can it become?**

Gustafson asks:

> **If I have more resources, how much larger a problem can I solve in roughly the same time?**

Gustafson's scaled-speedup model:

\[
S_G(P)=P-f_s(P-1)
\]

### Predict

For a 5% serial fraction and 32 processors, which model will give the larger number? Why?

In [ ]:
serial_fraction = 0.05
core_counts = [1, 2, 4, 8, 16, 32, 64]

amdahl_values = [amdahl_speedup(p, serial_fraction) for p in core_counts]
gustafson_values = [p - serial_fraction * (p - 1) for p in core_counts]

print(f"At 32 cores with {serial_fraction:.0%} serial work:")
print(f"  Amdahl fixed-size speedup:     {amdahl_speedup(32, serial_fraction):.2f}x")
print(f"  Gustafson scaled speedup:      {32 - serial_fraction*(32-1):.2f}x")

plt.figure(figsize=(7, 4))
plt.plot(core_counts, amdahl_values, marker='o', label='Amdahl: fixed problem')
plt.plot(core_counts, gustafson_values, marker='s', label='Gustafson: scaled problem')
plt.xlabel('Cores')
plt.ylabel('Modelled speedup')
plt.title('Same machine, different question')
plt.xticks(core_counts)
plt.grid(alpha=0.25)
plt.legend()
plt.show()

<details>
<summary><strong>Reveal answer</strong></summary>

Gustafson gives the larger value because it is **not predicting the same experiment**.

- **Amdahl:** fixed total problem size; extra processors divide the same parallel work.
- **Gustafson:** problem size grows with available resources; the parallel part grows while the serial part is treated as roughly fixed.

Do not ask which law is “better.” Ask **which question matches the experiment**.
</details>

# 4 — Strong scaling vs weak scaling

### Strong scaling

Keep the **total problem size fixed** and add workers.

Question: *How much faster does the same problem finish?*

### Weak scaling

Keep the **work per worker approximately fixed** as workers are added.

Question: *Can the system handle a proportionally larger problem in roughly the same time?*

We will use short sleeps to simulate equal pieces of computational work.

### Predict

1. For strong scaling, what should happen to runtime as workers increase?
2. For weak scaling, what should happen to runtime if scaling is ideal?

In [ ]:
UNIT_WORK = 0.04
WORKERS = [1, 2, 4, 8]

def simulated_unit(_):
    time.sleep(UNIT_WORK)
    return 1

def run_tasks(n_tasks, n_workers):
    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        list(pool.map(simulated_unit, range(n_tasks)))
    return time.perf_counter() - start

# Strong scaling: same total work (16 tasks)
strong_times = [run_tasks(16, p) for p in WORKERS]

# Weak scaling: same work per worker (4 tasks per worker)
weak_times = [run_tasks(4 * p, p) for p in WORKERS]

print('Strong scaling — fixed 16 tasks')
for p, t in zip(WORKERS, strong_times):
    print(f'{p:2d} workers -> {t:.3f} s')

print('\nWeak scaling — 4 tasks per worker')
for p, t in zip(WORKERS, weak_times):
    print(f'{p:2d} workers -> {t:.3f} s')

plt.figure(figsize=(7, 4))
plt.plot(WORKERS, strong_times, marker='o', label='Strong scaling runtime')
plt.plot(WORKERS, weak_times, marker='s', label='Weak scaling runtime')
plt.xlabel('Workers')
plt.ylabel('Runtime (s)')
plt.title('Strong vs weak scaling — conceptual experiment')
plt.xticks(WORKERS)
plt.grid(alpha=0.25)
plt.legend()
plt.show()

### Observe and explain

- Did strong-scaling runtime fall as workers increased?
- Did weak-scaling runtime stay approximately constant?
- Why are the measurements not perfectly ideal?

<details>
<summary><strong>Reveal answer</strong></summary>

In the strong-scaling experiment, the **same 16 tasks** are shared across more workers, so runtime should fall.

In the weak-scaling experiment, every worker receives the same amount of work, so total work grows with worker count while runtime should remain roughly constant.

Small deviations come from Python/thread scheduling and timing overhead. This experiment illustrates the definitions; it is not a machine benchmark.
</details>

# 5 — Scaling detective
### Four curves, four stories

A scaling graph is evidence. Your job is to decide what story it supports.

For each case below, ask:

1. Does performance keep improving?
2. Does it plateau?
3. Does it become worse after some point?
4. What **possible** cause is consistent with the curve?

Possible explanations:

- good scaling;
- serial fraction;
- communication/synchronization overhead;
- load imbalance.

> A graph can suggest a bottleneck, but it does not prove the exact cause.

In [ ]:
detective_cores = [1, 2, 4, 8, 16, 32]

curves = {
    'A': [1.0, 1.95, 3.75, 7.0, 12.5, 20.0],
    'B': [1.0, 1.82, 3.08, 4.71, 6.40, 7.80],
    'C': [1.0, 1.90, 3.60, 6.10, 7.00, 5.80],
    'D': [1.0, 1.70, 2.90, 4.10, 5.00, 5.50],
}

for label, values in curves.items():
    plt.figure(figsize=(6.6, 3.6))
    plt.plot(detective_cores, values, marker='o', label=f'Curve {label}')
    plt.plot(detective_cores, detective_cores, linestyle='--', alpha=0.5, label='Ideal')
    plt.xlabel('Cores')
    plt.ylabel('Speedup')
    plt.title(f'Scaling detective — Curve {label}')
    plt.xticks(detective_cores)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()

### Your diagnosis

Before revealing the answer, write one short hypothesis for each curve:

- **A:**
- **B:**
- **C:**
- **D:**

<details>
<summary><strong>Reveal answer</strong></summary>

One reasonable interpretation is:

- **A — good but non-ideal scaling:** performance continues to improve substantially.
- **B — serial-fraction limit:** speedup rises but increasingly plateaus.
- **C — overhead dominates at high core count:** performance improves, peaks, then becomes worse.
- **D — persistent inefficiency / possible load imbalance:** more cores help, but a large part of the machine is not translating into useful speedup.

These are **diagnostic hypotheses**, not proofs. Profiling and further measurements would be needed to establish the exact cause.
</details>

# 6 — When should we stop adding cores?
### Correct a misleading claim

A fixed workload was measured as follows:

| Cores | Runtime |
|---:|---:|
| 1 | 160 s |
| 2 | 84 s |
| 4 | 46 s |
| 8 | 28 s |
| 16 | 24 s |
| 32 | 26 s |

A colleague says:

> **“32 cores must be best because it uses the most compute.”**

### Predict

Is that claim supported by the data? What would you report instead?

In [ ]:
cores = [1, 2, 4, 8, 16, 32]
times = [160.0, 84.0, 46.0, 28.0, 24.0, 26.0]

baseline = times[0]
speedups = [baseline / t for t in times]
efficiencies = [s / p for s, p in zip(speedups, cores)]

best_index = min(range(len(times)), key=times.__getitem__)

print(f"Fastest measured configuration: {cores[best_index]} cores")
print(f"Fastest runtime:                {times[best_index]:.1f} s")
print()
print(f"{'Cores':>5} {'Time(s)':>8} {'Speedup':>9} {'Efficiency':>12}")
for p, t, s, e in zip(cores, times, speedups, efficiencies):
    print(f"{p:5d} {t:8.1f} {s:9.2f} {100*e:11.1f}%")

plt.figure(figsize=(7, 4))
plt.plot(cores, times, marker='o')
plt.xlabel('Cores')
plt.ylabel('Runtime (s)')
plt.title('More cores can eventually make a fixed workload slower')
plt.xticks(cores)
plt.grid(alpha=0.25)
plt.show()

<details>
<summary><strong>Reveal answer</strong></summary>

The claim is **not supported**.

For this fixed workload and these measurements:

- **16 cores** give the lowest observed runtime.
- Moving from 16 to 32 cores makes the run **slower**.
- Efficiency also falls as cores are added.

A defensible statement is:

> “For this workload, runtime improved up to 16 cores, then degraded at 32 cores. Additional resources no longer improved time-to-solution.”

The graph alone does not prove whether the cause is communication, synchronization, memory pressure, load imbalance or another overhead. That is a profiling question for Session 13.
</details>

# 7 — Parallel patterns: recognize the structure

Match each real problem to the most natural parallel pattern.

### Problems

**A. Satellite-image preprocessing**  
Thousands of independent images receive the same transformation.

**B. Global energy total**  
Every process computes a local energy value; one global sum is required.

**C. Cumulative particle offsets**  
Each position depends on the sum of all previous counts.

**D. Weather grid**  
Each region updates local cells and exchanges boundary values with neighbouring regions.

**E. Data-processing workflow**  
Input → clean → transform → analyse → write output, with different items in different stages simultaneously.

### Choose from

**MAP / TASK FARM · REDUCE · SCAN · STENCIL / DOMAIN DECOMPOSITION · PIPELINE**

### Predict

Write your choices before running the next cell.

In [ ]:
answers = {
    'A': ('MAP / TASK FARM', 'Independent items receive the same operation.'),
    'B': ('REDUCE', 'Many local values are combined into one result.'),
    'C': ('SCAN', 'Prefix/cumulative results preserve a structured dependency.'),
    'D': ('STENCIL / DOMAIN DECOMPOSITION', 'Local updates depend on neighbouring boundary data.'),
    'E': ('PIPELINE', 'Different stages process different items concurrently.'),
}

for key, (pattern, reason) in answers.items():
    print(f'{key}: {pattern:<30} {reason}')

### Now connect pattern to scalability

For each pattern, ask what may eventually limit scaling:

- **Map / task farm:** task overhead, imbalance, shared I/O
- **Reduce:** global communication / synchronization
- **Scan:** dependency structure and synchronization
- **Stencil / domain decomposition:** halo exchange, memory bandwidth, surface-to-volume ratio
- **Pipeline:** slowest stage and pipeline imbalance

<details>
<summary><strong>Reveal key idea</strong></summary>

Parallel patterns are not only programming templates. They also suggest **where scalability limits are likely to appear**.

The same core count can behave very differently depending on the algorithm's communication, synchronization and dependency structure.
</details>

# 8 — Final challenge: make an evidence-based scaling claim

Use this checklist whenever someone says an application “scales well”:

1. **What is the baseline?**
2. **What metric is being reported?** Runtime, speedup, efficiency, throughput?
3. **Is the problem size fixed or growing?** Strong or weak scaling?
4. **How many resources were used?**
5. **Where do diminishing returns begin?**
6. **Does the graph support the claim?**
7. **What additional evidence would be needed to explain the limit?**

### One-sentence exit ticket

Complete:

> **Adding more processors helps when ________, but eventually ________ can limit scalability.**